# OpenCV Crash Course: 2026 Edition
## Module 10: ONNX Export & Deployment
**Estimated time: 15 minutes**

- Why export (PyTorch is for training, not deployment)
- Export YOLO to ONNX in one line
- Run it with ONNX Runtime and OpenCV DNN
- Compare speed

> In production you ship one small `.onnx` file, not a 2 GB PyTorch install.

In [ ]:
# !pip install ultralytics onnx onnxruntime

In [ ]:
import cv2
import matplotlib.pyplot as plt
import urllib.request, os, time
from ultralytics import YOLO

def show(img, title="", figsize=(9, 6)):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title, fontsize=13)
    plt.axis('off')
    plt.show()

os.makedirs('input', exist_ok=True)
if not os.path.exists('input/bus.jpg'):
    urllib.request.urlretrieve('https://ultralytics.com/images/bus.jpg', 'input/bus.jpg')
img = cv2.imread('input/bus.jpg')

## 1. Export

| Format | Runs on |
|---|---|
| **onnx** | anywhere (default choice) |
| openvino | Intel CPUs |
| engine (TensorRT) | NVIDIA GPU / Jetson |
| tflite | Android, Raspberry Pi |

In [ ]:
model = YOLO('models/yolo26n.pt')
onnx_path = model.export(format='onnx', imgsz=640, opset=12)
print(onnx_path, f"{os.path.getsize(onnx_path) / 1e6:.1f} MB")

## 2. Run the ONNX Model

Easiest: Ultralytics loads `.onnx` with the same API.

In [ ]:
onnx_model = YOLO(onnx_path, task='detect')
show(onnx_model(img, verbose=False)[0].plot(), 'Same result, ONNX model')

Without Ultralytics, the model needs a **blob** `(1, 3, 640, 640)` and returns `(1, 84, 8400)`:
8400 candidate boxes x (4 box values + 80 class scores). Then NMS (`cv2.dnn.NMSBoxes`) removes duplicates.

In [ ]:
import onnxruntime as ort

blob = cv2.dnn.blobFromImage(img, 1 / 255.0, (640, 640), swapRB=True)

session = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
inp = session.get_inputs()[0].name
print("ONNX Runtime output:", session.run(None, {inp: blob})[0].shape)

net = cv2.dnn.readNetFromONNX(onnx_path)       # only opencv-python needed
net.setInput(blob)
print("OpenCV DNN output  :", net.forward().shape)

## 3. Speed Comparison

Always measure on the machine you deploy to.

In [ ]:
def bench(fn, n=20):
    fn()
    t0 = time.time()
    for _ in range(n):
        fn()
    return (time.time() - t0) / n * 1000

def run_cv():
    net.setInput(blob)
    net.forward()

for name, fn in [('PyTorch', lambda: model(img, verbose=False)),
                 ('ONNX Runtime', lambda: session.run(None, {inp: blob})),
                 ('OpenCV DNN', run_cv)]:
    ms = bench(fn)
    print(f"{name:<13} {ms:6.1f} ms  (~{1000 / ms:.0f} FPS)")

---
## Summary

| Task | Code |
|------|------|
| Export | `model.export(format='onnx')` |
| Load ONNX | `YOLO('models/yolo26n.onnx', task='detect')` |
| ONNX Runtime | `ort.InferenceSession(path).run(None, {name: blob})` |
| OpenCV DNN | `cv2.dnn.readNetFromONNX(path)` |

### Course complete
OpenCV basics → detect → track & count → train → segment → OCR → deploy.